# 🧠 SVOMPTR-9B: Master Neural Distillation & MoE Training

This notebook covers:
- 🛡️ Google Drive Integration (`svomptr_brain`)
- 🗃️ Massive 5M Data Generation via vLLM
- 🧺 Fast Knowledge Distillation & DOP Alignment
- 🔄 Auto-resume & Idle Prevention
- 🚀 LocalTunnel ML API Deployment

### 🛡️ 1. Prevent Idle Timeout
Press `Ctrl+Shift+I` (or `Cmd+Option+I` on Mac) to open browser Developer Tools. Go to the **Console** tab, paste the following code, and press Enter. This will keep Colab alive during long training sessions.

```javascript
function KeepClicking(){
  console.log("Keeping Colab active...");
  document.querySelector("colab-toolbar-button").click();
}
setInterval(KeepClicking, 60000);
```

In [ ]:
# 🛡️ 2. Drive Mount & Brain Initialization
from google.colab import drive
import os

drive.mount('/content/drive')
brain_dir = '/content/drive/MyDrive/svomptr_brain'

# Create necessary directories
directories = ['datasets', 'checkpoints', 'weights', 'dop_alignment']
for d in directories:
    os.makedirs(os.path.join(brain_dir, d), exist_ok=True)

print(f"✅ SVOMPTR Brain active at: {brain_dir}")

In [ ]:
# ⚙️ 3. Install Core Dependencies
print("Installing core dependencies... This may take a few minutes.")
!pip install transformers accelerate bitsandbytes datasets sentence-transformers fastapi uvicorn pydantic pyngrok vllm trl

import torch
if torch.cuda.is_available():
    print("✅ GPU Detected. Fine-tuning and high-speed generation enabled.")
    !pip install faiss-gpu
else:
    print("⚠️ CPU Only Mode. Installing CPU-optimized FAISS and skipping vLLM.")
    !pip install faiss-cpu


In [ ]:
# 📂 4. Project Setup
import sys
import os
repo_path = '/content/svomptr-project'
if not os.path.exists(repo_path):
    print("Cloning SVOMPTR repository to /content/svomptr-project...")
    !git clone https://github.com/kkomyoeminaung/svomptr-9b-moe-model-.git /content/svomptr-project
sys.path.append(repo_path)

### 🗃️ 5. Massive Synthetic Data Generation (vLLM for Speed)
Generates millions of structured SVOMPTR English-Myanmar pairs rapidly. Resumes automatically if interrupted.

In [ ]:
%%writefile generate_data.py
import os
import json
import time
import torch
from google.colab import drive

TARGET_COUNT = 5_000_000
BRAIN_DIR = '/content/drive/MyDrive/svomptr_brain'
DATASET_FILE = os.path.join(BRAIN_DIR, 'datasets', f'synthetic_{TARGET_COUNT}.jsonl')
TEACHER_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"

def verify_drive():
    if not os.path.exists(BRAIN_DIR):
        print("❌ ERROR: Google Drive not mounted or brain path invalid!")
        return False
    return True

try:
    drive.mount('/content/drive', force_remount=False)
    if not torch.cuda.is_available():
        TEACHER_MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
        from transformers import pipeline
        pipe = pipeline('text-generation', model=TEACHER_MODEL_ID, device_map='cpu')
        USE_VLLM = False
        print(f"🐢 CPU Mode: Quality check using {TEACHER_MODEL_ID}")
    else:
        from vllm import LLM, SamplingParams
        llm = LLM(model=TEACHER_MODEL_ID, dtype="bfloat16", max_model_len=4096, enable_prefix_caching=True)
        sampling_params = SamplingParams(temperature=0.8, top_p=0.95, max_tokens=512)
        USE_VLLM = True
        print(f"🚀 GPU Mode: Accelerating {TEACHER_MODEL_ID} via vLLM")
except Exception as e:
    print(f"⚠️ Initialization Fallback: {e}")
    from transformers import pipeline
    pipe = pipeline('text-generation', model=TEACHER_MODEL_ID, device_map='auto')
    USE_VLLM = False

generated_count = 0
if os.path.exists(DATASET_FILE):
    with open(DATASET_FILE, 'r', encoding='utf-8') as f:
        generated_count = sum(1 for _ in f)
print(f"📈 Progress: {generated_count}/{TARGET_COUNT} ({(generated_count/TARGET_COUNT)*100:.2f}%)")

BATCH_SIZE = 5000 if USE_VLLM else 5
base_prompt = "Generate one complex English sentence and its precise Myanmar translation with detailed SVOMPTR analysis in JSON format."

while generated_count < TARGET_COUNT:
    if not verify_drive(): break
    t0 = time.time()
    current_batch = min(BATCH_SIZE, TARGET_COUNT - generated_count)
    
    if USE_VLLM:
        outputs_raw = llm.generate([base_prompt] * current_batch, sampling_params, use_tqdm=False)
        outputs = [out.outputs[0].text.strip() for out in outputs_raw]
    else:
        outputs_raw = pipe([base_prompt] * current_batch, max_new_tokens=400)
        outputs = [out[0]['generated_text'].split(base_prompt)[-1].strip() for out in outputs_raw]
    
    with open(DATASET_FILE, 'a', encoding='utf-8') as f:
        for text in outputs:
            try:
                if text.startswith('{') and text.endswith('}'):
                    f.write(text.replace('\n', ' ') + '\n')
                    generated_count += 1
            except: continue
    
    pace = current_batch / (time.time() - t0)
    print(f"✨ [{time.strftime('%H:%M:%S')}] Pace: {pace:.2f} items/s | Total: {generated_count}/{TARGET_COUNT}")


In [ ]:
# !python generate_data.py

### 🧺 6. Distillation & DOP Alignment Training
Train the Chat Expert student model using the synthetic dataset. Auto-saves to Drive and resumes on interruption.

In [ ]:
%%writefile train_distillation.py
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from datasets import load_dataset

BRAIN_DIR = '/content/drive/MyDrive/svomptr_brain'
DATASET_FILE = os.path.join(BRAIN_DIR, 'datasets', 'synthetic_5000000.jsonl')
CKPT_DIR = os.path.join(BRAIN_DIR, 'checkpoints', 'chat_expert')
FINAL_DIR = os.path.join(BRAIN_DIR, 'weights', 'chat_expert_final')

print(f"🧺 [Distillation] Targeting Dataset: {DATASET_FILE}")

if not os.path.exists(DATASET_FILE):
    print("⚠️ dataset not found. Please run generate_data.py first!")
    exit(1)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True
)

def formatting_func(example):
    try:
        en = example.get('en', '')
        my = example.get('my', '')
        s = example.get('svomptr_structure', {})
        struct = f"S:{s.get('S','')}|V:{s.get('V','')}|O:{s.get('O','')}"
        
        text = f"<|im_start|>user\nTranslate: {en}<|im_end|>\n<|im_start|>assistant\n{my} (Structure: {struct})<|im_end|>"
        return {"text": text}
    except: return {"text": ""}

print("📦 [Distillation] Loading dataset into memory-mapped stream...")
dataset = load_dataset("json", data_files=DATASET_FILE, split="train")
dataset = dataset.map(formatting_func, remove_columns=dataset.column_names)

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    num_train_epochs=1, # 5M items usually only need 1 pass for distillation
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    save_strategy="steps",
    save_steps=1000,
    logging_steps=100,
    bf16=True,
    save_total_limit=2,
    report_to="none",
    resume_from_checkpoint=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    dataset_text_field="text" # For compatibility if using SFTTrainer style
)

last_checkpoint = None
if os.path.isdir(CKPT_DIR):
    ckpts = [os.path.join(CKPT_DIR, d) for d in os.listdir(CKPT_DIR) if d.startswith("checkpoint")]
    if ckpts:
        last_checkpoint = max(ckpts, key=os.path.getmtime)
        print(f"🔄 [Distillation] Resuming from: {last_checkpoint}")

print("🚀 [Distillation] Starting Training Engine...")
trainer.train(resume_from_checkpoint=last_checkpoint)

print(f"✅ [Distillation] Success! Saving weights to {FINAL_DIR}")
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)

# Save DOP signature
with open(os.path.join(dop_dir, "alignment_signature.txt"), "w") as f:
    f.write("DOP Alignment: SUCCESS\nModel: Qwen2.5-1.5B (Distilled)\n")


In [ ]:
# !python train_distillation.py

### 🤖 6.5. Train Router & Sub-Experts (Full MoE)
Once the main chat expert is distilled, train the semantic router and the sub-domain specialists.

In [ ]:
import sys
if '/content/svomptr-project' not in sys.path:
    sys.path.insert(0, '/content/svomptr-project')

from svomptr_moe.train_sub_experts import train_domain_experts
from svomptr_moe.train_router import train_router

# Set brain path to ensure weights map to Google Drive
import os
os.environ['SVOMPTR_BRAIN_PATH'] = '/content/drive/MyDrive/svomptr_brain'

print("Training Sub-Experts...")
# train_domain_experts()
print("Training Router...")
# train_router()


### 🚀 7. Run ML API (LocalTunnel)
Launch the FastAPI backend serving the weights from `svomptr_brain`.

In [ ]:
!npm install -g localtunnel
import subprocess
import time

print("Starting SVOMPTR FastAPI server...")
# Make sure to set SVOMPTR_BRAIN_PATH inside your repo if you want it to load weights from Drive
import os
os.environ["SVOMPTR_BRAIN_PATH"] = "/content/drive/MyDrive/svomptr_brain"

api_proc = subprocess.Popen(["uvicorn", "ml_api:app", "--host", "0.0.0.0", "--port", "8000", "--reload", "--app-dir", "/content/svomptr-project"])
time.sleep(5)

print("Starting LocalTunnel... Copy the URL below and set VITE_ML_API_URL and ML_API_URL in your Next.js/Vite frontend!")
!lt --port 8000 --subdomain svogateway9b